# create_deep_agent와 도구

생성 함수의 인자와 기본 도구 표면을 설치된 0.7.0에서 확인합니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

고정 수업 자료를 반환하는 course_lookup 도구와 에이전트를 만듭니다.

In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from deepagents import create_deep_agent

@tool
def course_lookup(chapter: int) -> str:
    """Return the fixed course topic for one chapter."""
    return {27: 'DeepAgents 하네스'}[chapter]

# 모델과 한 개의 수업 조회 도구를 가진 에이전트를 만듭니다.
model = init_chat_model('openai:gpt-5.4-mini')
agent = create_deep_agent(model=model, tools=[course_lookup])
print('그래프 노드:', sorted(agent.get_graph().nodes))

그래프 노드: ['PatchToolCallsMiddleware.before_agent', '__end__', '__start__', 'model', 'tools']


## 단계 2

모델에게 course_lookup 호출을 요청하고 메시지 순서를 출력합니다.

In [4]:
# 모델이 course_lookup을 선택하도록 구체적인 질문을 보냅니다.
result = agent.invoke({'messages':[{'role':'user','content':'course_lookup으로 Chapter 27 주제를 확인한 뒤 한 문장으로 답하세요.'}]})
for message in result['messages']:
    print(type(message).__name__, getattr(message, 'name', None))

HumanMessage None
AIMessage None
ToolMessage course_lookup
AIMessage None


## 단계 3

AI 도구 호출과 course_lookup ToolMessage가 모두 있는지 확인합니다.

In [5]:
# AI 도구 호출과 도구 반환 메시지가 모두 있는지 확인합니다.
calls = [call for m in result['messages'] for call in getattr(m, 'tool_calls', [])]
tool_messages = [m for m in result['messages'] if type(m).__name__ == 'ToolMessage']
print('호출 도구:', [call['name'] for call in calls])
print('도구 결과:', [m.content for m in tool_messages if m.name == 'course_lookup'])
assert any(call['name'] == 'course_lookup' for call in calls)
assert any(m.name == 'course_lookup' and 'DeepAgents' in str(m.content) for m in tool_messages)

호출 도구: ['course_lookup']
도구 결과: ['DeepAgents 하네스']
